# Cụm theo luật với ba bộ nối quan hệ

Hai nguồn cụm × ba bộ nối tạo sáu cấu hình. Trình chạy chấm cả sáu trên validation và chấm cấu hình đã chọn với đối chứng nối gần nhất trên test.

[Báo cáo tổng hợp](reports/tong-hop-pipeline-smartnotice-v3.3-2026-11-23.md) · [Giao thức đánh giá](../../docs/evaluation.md)

## 1. Cấu hình và dữ liệu

Nạp thông báo đã chuẩn bị và kiểm tra hash, nhóm, schema và các khoảng ký tự. Tham chiếu test v3.3 là chú giải lịch sử có máy hỗ trợ.

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

repo = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "tools/compare_notice_pipelines.py").is_file()),
    None,
)
if repo is None:
    raise RuntimeError("Mở notebook từ workspace SmartNotice.")
os.chdir(repo)
split = Path(os.environ.get("SMARTNOTICE_SPLIT_ROOT", "tmp/drive-run-20261123-fresh")).resolve()
base = Path(os.environ.get("SMARTNOTICE_BAMIBERT_DIR", "tmp/bamibert-checkpoint")).resolve()
destination = Path(os.environ.get(
    "SMARTNOTICE_NOTEBOOK_OUTPUT", "reports/generated/pipeline-notebooks-v33-20261123"
)).resolve()
baseline = Path(os.environ.get(
    "SMARTNOTICE_BASELINE", "reports/generated/bamibert/v3_3-2026-11-23-final-seed42"
)).resolve()


def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))


def run_tool(script, *arguments, **environment):
    env = {**os.environ, "PYTHONHASHSEED": "42", "TOKENIZERS_PARALLELISM": "false", **environment}
    result = subprocess.run(
        [sys.executable, "-u", str(repo / "tools" / script), *map(str, arguments)],
        cwd=repo, env=env, capture_output=True, text=True, encoding="utf-8", errors="replace",
    )
    if result.returncode:
        print(result.stdout[-4000:], result.stderr[-4000:])
        result.check_returncode()
    print(result.stdout.strip())

In [ ]:
from smartnotice.modeling.data import labeled_links, load_development, sha256, span_key

development, provenance = load_development(split)
for part, docs in development.items():
    print(part, {
        "notices": len(docs),
        "groups": len({d.group_id for d in docs}),
        "typed_spans": sum(len({span_key(s) for s in d.annotation.spans}) for d in docs),
        "links": sum(len(set(labeled_links(d.annotation))) for d in docs),
        "events": sum(len(d.annotation.events) for d in docs),
    })
print("Phân bổ:", provenance["split_counts"], "gold_ready:", provenance["gold_ready"])

## 2. Xây luật và từ điển rồi huấn luyện đầu nối

Encoder LoRA gốc giữ cố định. Đầu nối học lại 12 epoch, với mẫu âm từ cặp cụm chú giải và cặp do luật sinh.

In [ ]:
out = destination / "run-07"
args = ("--split", split, "--base", base, "--baseline", baseline, "--output", out)
run_tool("evaluate_rule_bami.py", "select", *args)

## 3. Kiểm tra điểm validation và khóa cấu hình

In [ ]:
def summarize_configs(scores):
    for name, score in sorted(scores.items()):
        print(name, "F1 cụm", f"{score['span']['micro']['f1']:.2%}",
              "F1 quan hệ", f"{score['link']['micro']['f1']:.2%}",
              "F1 sự kiện", f"{score['event']['f1']:.2%}")


selection = read_json(out / "selection.json")
print("Cấu hình đã khóa:", selection["winner"])
summarize_configs(read_json(out / "select.json"))

## 4. Suy luận test và đối chiếu bộ nối trên cùng đầu mút

In [ ]:
run_tool("evaluate_rule_bami.py", "test", *args)
scores = read_json(out / "test.json")
summarize_configs(scores)
for name, score in scores.items():
    print(name, "đủ đầu mút theo luật", score["reference_links_with_both_rule_spans"],
          "biểu diễn được", score["reference_links_with_both_encodable_spans"],
          "liên kết đúng", score["link"]["micro"]["matched"])